## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [1]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


## <font color='blue'> WRITE CODE BELOW </font>

In [2]:
def q1(reviews):
    return (reviews.isna().mean() * 100).round(2)

def q2(products):
    return (products.isna().mean() * 100).round(2)

def q3(reviews, products):
    reviews_numeric = reviews.assign(
        overall=dd.to_numeric(reviews["overall"], errors="coerce")
    )

    products_numeric = products.assign(
        price=dd.to_numeric(products["price"], errors="coerce")
    )

    review_ratings = reviews_numeric[["asin", "overall"]].dropna(
        subset=["asin", "overall"]
    )

    product_prices = products_numeric[["asin", "price"]].dropna(
        subset=["asin", "price"]
    )

    merged = review_ratings.merge(product_prices, on="asin", how="inner")

    return merged["price"].corr(merged["overall"])

def q4(products):
    price = dd.to_numeric(products["price"], errors="coerce")
    return price.dropna()

def q5(products):
    def extract_super_category(value):
        if pd.isna(value):
            return np.nan

        value = str(value)

        match = re.search(r"""['"]([^'"]+)['"]""", value)

        if match:
            return match.group(1)

        return np.nan

    super_categories = products["categories"].map_partitions(
        lambda s: s.map(extract_super_category),
        meta=("categories", "object")
    )

    return super_categories.dropna().value_counts()

from dask.distributed import get_client, as_completed, wait

def clean_asin_column(ddf, column_name="asin"):
    cleaned = ddf[[column_name]].dropna(subset=[column_name])
    cleaned = cleaned.astype({column_name: "object"})
    cleaned = cleaned.drop_duplicates()

    if column_name != "asin":
        cleaned = cleaned.rename(columns={column_name: "asin"})

    return cleaned


def add_product_exists_marker(product_asins):
    def add_marker_partition(pdf):
        pdf = pdf.copy()
        pdf["product_exists"] = True
        return pdf

    return product_asins.map_partitions(
        add_marker_partition,
        meta=pd.DataFrame({
            "asin": pd.Series(dtype="object"),
            "product_exists": pd.Series(dtype="bool")
        })
    )


def early_stop_boolean_partitions(partition_checks, label):
    client = get_client()

    print(f"[{label}] Converting partition checks to delayed tasks...")
    delayed_checks = partition_checks.to_delayed()

    print(f"[{label}] Number of partition-check tasks: {len(delayed_checks)}")
    print(f"[{label}] Submitting tasks in parallel...")

    futures = client.compute(delayed_checks)
    completed = 0

    try:
        for future in as_completed(futures):
            completed += 1
            result = future.result()

            found = bool(result["has_dangling"].iloc[0])

            print(
                f"[{label}] Completed {completed}/{len(futures)} "
                f"| dangling found: {found}"
            )

            if found:
                print(f"[{label}] Dangling reference found. Cancelling remaining tasks...")
                client.cancel(futures)
                return 1

        print(f"[{label}] Checked all partitions. No dangling references found.")
        return 0

    finally:
        client.cancel(futures)


def check_joined_partitions_for_dangling(joined, label):
    def partition_has_dangling(pdf):
        if pdf.empty:
            return pd.DataFrame({
                "has_dangling": pd.Series([False], dtype="bool")
            })

        has_dangling = pdf["product_exists"].isna().any()

        return pd.DataFrame({
            "has_dangling": pd.Series([bool(has_dangling)], dtype="bool")
        })

    partition_checks = joined[["product_exists"]].map_partitions(
        partition_has_dangling,
        meta=pd.DataFrame({
            "has_dangling": pd.Series(dtype="bool")
        })
    )

    return early_stop_boolean_partitions(partition_checks, label)

def q6(reviews, product_asins_with_marker):
    label = "Q6"
    print(f"[{label}] Starting reviews.asin -> products.asin dangling check...")
    start = time.time()

    print(f"[{label}] Cleaning and deduplicating reviews.asin...")
    review_asins = clean_asin_column(reviews, "asin")

    print(f"[{label}] Building distributed left join on asin...")
    joined = review_asins.merge(
        product_asins_with_marker,
        on="asin",
        how="left"
    )

    print(f"[{label}] Joined columns:", list(joined.columns))

    ans = check_joined_partitions_for_dangling(joined, label)

    end = time.time()
    print(f"[{label}] Finished with answer: {ans}")
    print(f"[{label}] Time taken: {end - start:.2f} seconds")

    return ans

def q7(products, product_asins_with_marker):
    label = "Q7"
    print(f"[{label}] Starting products.related -> products.asin dangling check...")
    start = time.time()

    asin_pattern = re.compile(r"""['"]([A-Za-z0-9]{10})['"]""")

    def extract_related_asins_partition(pdf):
        related_asins = []

        for value in pdf["related"].dropna():
            refs = asin_pattern.findall(str(value))

            if refs:
                related_asins.extend(refs)

        return pd.DataFrame({
            "asin": pd.Series(related_asins, dtype="object")
        })

    print(f"[{label}] Extracting related ASINs in parallel...")

    related_asins = products[["related"]].map_partitions(
        extract_related_asins_partition,
        meta=pd.DataFrame({
            "asin": pd.Series(dtype="object")
        })
    )

    print(f"[{label}] Dropping duplicate related ASINs...")

    related_asins = related_asins.dropna(subset=["asin"])
    related_asins = related_asins.astype({"asin": "object"})
    related_asins = related_asins.drop_duplicates()

    print(f"[{label}] Building distributed left join on asin...")

    joined = related_asins.merge(
        product_asins_with_marker,
        on="asin",
        how="left"
    )

    print(f"[{label}] Joined columns:", list(joined.columns))

    ans = check_joined_partitions_for_dangling(joined, label)

    end = time.time()
    print(f"[{label}] Finished with answer: {ans}")
    print(f"[{label}] Time taken: {end - start:.2f} seconds")

    return ans

In [ ]:
### read in the user_reviews_path and products_path files, perform your calculations and place the answers in variables ans1 - ans7.
def PA1(user_reviews_path, products_path):
    reviews = dd.read_csv(
        user_reviews_path,
        dtype="object",
        assume_missing=True,
        blocksize="256MB"
    )

    products = dd.read_csv(
        products_path,
        dtype="object",
        assume_missing=True,
        blocksize="256MB"
    )

    print("[PA1] Read reviews and products csvs")
    
    q1_lazy = q1(reviews)
    q2_lazy = q2(products)
    q3_lazy = q3(reviews, products)
    q4_lazy = q4(products)
    q5_lazy = q5(products)

    print("[PA1] Scheduling Q1-5 lazy compute")

    q1_res, q2_res, q3_res, q4_res, q5_res = dask.compute(
        q1_lazy,
        q2_lazy,
        q3_lazy,
        q4_lazy,
        q5_lazy
    )

    ans1 = {
        k: float(v)
        for k, v in q1_res.to_dict().items()
    }

    ans2 = {
        k: float(v)
        for k, v in q2_res.to_dict().items()
    }

    ans3 = round(float(q3_res), 2)

    q4_res_stats = q4_res.describe()

    ans4 = {
        "mean": float(q4_res_stats["mean"]),
        "std": float(q4_res_stats["std"]),
        "min": float(q4_res_stats["min"]),
        "max": float(q4_res_stats["max"]),
        "median": float(q4_res_stats["50%"])
    }

    # don't use Dask median as this is an approximation,
    # use preferred pandas median
    # ans4 = {
    #     "mean": float(q4_res.mean()),
    #     "std": float(q4_res.std()),
    #     "min": float(q4_res.min()),
    #     "max": float(q4_res.max()),
    #     "median": float(q4_res.median())
    # }

    q5_res = q5_res.sort_values(ascending=False)
    ans5 = {
        str(k): int(v)
        for k, v in q5_res.to_dict().items()
    }

    print("[PA1] Finished Q1-5")

    print("[PA1] Building shared product ASIN table for Q6 and Q7...")

    product_asins = clean_asin_column(products, "asin")
    product_asins_with_marker = add_product_exists_marker(product_asins)

    print("[PA1] Persisting shared product ASIN table...")
    product_asins_with_marker = product_asins_with_marker.persist()
    wait(product_asins_with_marker)

    print("[PA1] Finished persisting shared product ASIN table.")

    print("[PA1] Running Q6...")
    ans6 = q6(reviews, product_asins_with_marker)
    print(f"[PA1] Q6 answer: {ans6}")

    print("[PA1] Running Q7...")
    ans7 = q7(products, product_asins_with_marker)
    print(f"[PA1] Q7 answer: {ans7}")

    return ans1, ans2, ans3, ans4, ans5, ans6, ans7

## <font color='red'> DO NOT MODIFY </font>

In [4]:
start = time.time()
ans1, ans2, ans3, ans4, ans5, ans6, ans7 = PA1(user_reviews_path="user_reviews.csv", products_path="products.csv")
end = time.time()
print(f"execution time = {end-start}s")

[PA1] Read reviews and products csvs
[PA1] Scheduling Q1-5 lazy compute
[PA1] Finished Q1-5
[PA1] Building shared product ASIN table for Q6 and Q7...
[PA1] Persisting shared product ASIN table...
[PA1] Finished persisting shared product ASIN table.
[PA1] Running Q6...
[Q6] Starting reviews.asin -> products.asin dangling check...
[Q6] Cleaning and deduplicating reviews.asin...
[Q6] Building distributed left join on asin...
[Q6] Joined columns: ['asin', 'product_exists']
[Q6] Converting partition checks to delayed tasks...
[Q6] Number of partition-check tasks: 79
[Q6] Submitting tasks in parallel...
[Q6] Completed 1/79 | dangling found: True
[Q6] Dangling reference found. Cancelling remaining tasks...
[Q6] Finished with answer: 1
[Q6] Time taken: 45.79 seconds
[PA1] Q6 answer: 1
[PA1] Running Q7...
[Q7] Starting products.related -> products.asin dangling check...
[Q7] Extracting related ASINs in parallel...
[Q7] Dropping duplicate related ASINs...
[Q7] Building distributed left join on a

In [7]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         